# SNAI2 motif scramble predictions

For each motif (`unique_id`), predict signal from 3 chrombpnet models (wt / dtag_3h / dtag_1d)
on both the WT and scrambled (mutant) 2114bp sequences. Additionally, calculate contribution changes across the motifs along the enhancer.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import get_custom_objects
import chrombpnet.training.utils.losses as losses
import chrombpnet.training.utils.one_hot as one_hot

# ---- Required for SHAP ----
import shap
import chrombpnet.evaluation.interpret.shap_utils as shap_utils
tf.compat.v1.disable_eager_execution()

2026-08-19 10:12:13.163753: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-08-19 10:12:13.163775: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
#1: Load models
def load_model_wrapper(model_h5):
    custom_objects = {"multinomial_nll": losses.multinomial_nll, "tf": tf}
    get_custom_objects().update(custom_objects)
    model = load_model(model_h5, compile=False)
    return model

# Model condition names are prefixed "model_" throughout to avoid any confusion
# with the "wt" / "mutant" *sequence* type (unrelated to which model this is).
MODEL_PATHS = {
    "model_wt":      "/storage/kaelanb/analysis/lucia/model/snail_modisco_test/all_peaks/models/chrombpnet_nobias.h5",
    "model_dtag_3h": "/storage/kaelanb/analysis/lucia/model/snail_modisco_test/all_peaks/dtag_3h/all_peaks/models/chrombpnet_nobias.h5",
    "model_dtag_1d": "/storage/kaelanb/analysis/lucia/model/snail_modisco_test/all_peaks/dtag_1d/all_peaks/models/chrombpnet_nobias.h5",
}

models = {name: load_model_wrapper(path) for name, path in MODEL_PATHS.items()}
print("Loaded models:", list(models.keys()))


2026-08-19 10:12:47.998814: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2026-08-19 10:12:47.998844: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2026-08-19 10:12:47.998875: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (badger.stanford.edu): /proc/driver/nvidia/version does not exist
2026-08-19 10:12:47.999157: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loaded models: ['model_wt', 'model_dtag_3h', 'model_dtag_1d']


In [3]:
#2: Build a SHAP counts-contribution explainer for each model
explainers = {}
for model_name, model in models.items():
    explainers[model_name] = shap.explainers.deep.TFDeepExplainer(
        (model.input, tf.reduce_sum(model.outputs[1], axis=-1)),
        shap_utils.shuffle_several_times,
        combine_mult_and_diffref=shap_utils.combine_mult_and_diffref)

print("Built explainers for:", list(explainers.keys()))


Built explainers for: ['model_wt', 'model_dtag_3h', 'model_dtag_1d']


In [4]:
#3: Load sequences + motif positions from the R-generated metadata
metadata_path = "/storage/kaelanb/analysis/lucia/analysis/custom_seq/snai2_scramble_metadata.csv"

df = pd.read_csv(metadata_path)

assert df["wt_seq"].str.len().eq(2114).all(), "All wt_seq must be 2114bp"
assert df["mutant_seq"].str.len().eq(2114).all(), "All mutant_seq must be 2114bp"
assert not df["unique_id"].duplicated().any(), "unique_id must be unique"

print(f"Loaded {len(df)} motifs")
df.head()


Loaded 63 motifs


,unique_id,wt_header,scramble_header,chrom,strand,genomic_start,genomic_end,window_start,window_end,motif_start_in_window,motif_end_in_window,motif_start_in_output,motif_end_in_output,motif_seq_wt,motif_seq_scrambled,wt_seq,mutant_seq
0,1,wt_1,scramble_1,chr11,-,8708967,8708973,8707914,8710027,1054,1060,497,503,CCACCTT,ATCGAGG,TTCTGGCTCCTTTTCCTTGGGAACGGAGGAGTAAGGACTGCCCTTC...,TTCTGGCTCCTTTTCCTTGGGAACGGAGGAGTAAGGACTGCCCTTC...
1,2,wt_2,scramble_2,chr20,+,17169684,17169690,17168631,17170744,1054,1060,497,503,GCACCCA,ATTATAC,TGGGGCAAAGCCTCTCCTCTCTGGGTCACATTTCAAGGAGGAGCTG...,TGGGGCAAAGCCTCTCCTCTCTGGGTCACATTTCAAGGAGGAGCTG...
2,3,wt_3,scramble_3,chr11,+,8709104,8709110,8708051,8710164,1054,1060,497,503,CAGGTGC,GCAAATA,GCAGGGTCTCCCAGCACCAGGTACCAGGCTCCTGCCAGGCTCCACA...,GCAGGGTCTCCCAGCACCAGGTACCAGGCTCCTGCCAGGCTCCACA...
3,4,wt_4,scramble_4,chr11,+,12319567,12319573,12318514,12320627,1054,1060,497,503,CAGGTGC,GGCTGAG,CTCTCTCACTAGGGTGAATAGTTGAAAATTTTAAGGAGGAAAAGGC...,CTCTCTCACTAGGGTGAATAGTTGAAAATTTTAAGGAGGAAAAGGC...
4,5,wt_5,scramble_5,chr11,+,12321352,12321358,12320299,12322412,1054,1060,497,503,CAGGTGT,ATCAGTA,CCGTGCCAGCTGGGGAGATGGCAGACTTCTTTTCTCAATGAACATT...,CCGTGCCAGCTGGGGAGATGGCAGACTTCTTTTCTCAATGAACATT...


In [5]:
#4: One-hot encode wt and mutant sequences
one_hot_seqs = {
    "wt":      one_hot.dna_to_one_hot(df["wt_seq"].tolist()),
    "mutant":  one_hot.dna_to_one_hot(df["mutant_seq"].tolist()),
}

print("WT one-hot shape:     ", one_hot_seqs["wt"].shape)
print("Mutant one-hot shape: ", one_hot_seqs["mutant"].shape)


WT one-hot shape:      (63, 2114, 4)
Mutant one-hot shape:  (63, 2114, 4)


In [6]:
#5: Predict with all 3 models, on both sequence types
def softmax(x, temp=1):
    norm_x = x - np.mean(x, axis=1, keepdims=True)
    return np.exp(temp * norm_x) / np.sum(np.exp(temp * norm_x), axis=1, keepdims=True)

def get_predictions(model, one_hot_seqs_arr):
    pred_logits, pred_logcts = model.predict(one_hot_seqs_arr)
    return softmax(pred_logits) * np.expand_dims(np.exp(pred_logcts)[:, 0], axis=1)

# predictions[model_name][seq_type] -> array of shape (n_motifs, 1000)
predictions = {}
for model_name, model in models.items():
    predictions[model_name] = {}
    for seq_type, oh in one_hot_seqs.items():
        predictions[model_name][seq_type] = get_predictions(model, oh)
        print(f"{model_name} / {seq_type} predictions shape:", predictions[model_name][seq_type].shape)


/storage/kaelanb/anaconda3/envs/chrombpnet/lib/python3.8/site-packages/keras/engine/training_v1.py:2079: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


model_wt / wt predictions shape: (63, 1000)
model_wt / mutant predictions shape: (63, 1000)
model_dtag_3h / wt predictions shape: (63, 1000)
model_dtag_3h / mutant predictions shape: (63, 1000)
model_dtag_1d / wt predictions shape: (63, 1000)
model_dtag_1d / mutant predictions shape: (63, 1000)


In [12]:
#6: Plot profiles
plot_dir = "/storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/"
os.makedirs(plot_dir, exist_ok=True)

model_names = ["model_wt", "model_dtag_3h", "model_dtag_1d"]
seq_colors = {
    "wt":      "blue",
    "mutant":  "red",
}

for i, row in enumerate(df.itertuples()):
    uid = row.unique_id
    motif_start = row.motif_start_in_output
    motif_end   = row.motif_end_in_output

    # Shared y-axis across all 3 model rows for this motif.
    ymax = max(
        predictions[m][s][i].max()
        for m in model_names
        for s in seq_colors
    )
    ylim = (0, ymax * 1.05)

    fig, axes = plt.subplots(3, 1, figsize=(10, 12), sharex=True)
    for ax in axes[1:]:
        ax.sharey(axes[0])

    for ax, model_name in zip(axes, model_names):
        for seq_type, color in seq_colors.items():
            ax.plot(predictions[model_name][seq_type][i], color=color, label=seq_type)

        ax.axvspan(motif_start, motif_end, alpha=0.2, color="grey", label="motif")
        ax.set_ylim(ylim)
        ax.set_title(f"{uid} - {model_name}")
        ax.set_ylabel("Predicted signal")
        ax.legend()

    axes[-1].set_xlabel("Position (bp)")

    fig.suptitle(uid)
    plt.tight_layout()
    outfile = os.path.join(plot_dir, f"{uid}.pdf")
    plt.savefig(outfile, dpi=150)
    plt.close(fig)

    if (i + 1) % 10 == 0 or (i + 1) == len(df):
        print(f"[{i+1}/{len(df)}] plotted {uid}")

print("\nAll plots written to:", plot_dir)


[10/63] plotted 10
[20/63] plotted 20
[30/63] plotted 30
[40/63] plotted 40
[50/63] plotted 50
[60/63] plotted 60
[63/63] plotted 63

All plots written to: /storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/


In [17]:
#7: Choose which unique_ids to compute contribution scores for, and their plot windows
contrib_windows = {
    "2":  (450, 850),
    "8":  (400, 600),
    "23": (400, 550),
    "20":  (150, 650),
    "25":  (100, 600),
    "26":  (200, 700),
    "30":  (200, 900),
    "42":  (200, 700),
    "49":  (200, 800),
}
contrib_unique_ids = list(contrib_windows.keys())

contrib_idx = df.index[df["unique_id"].astype(str).isin(contrib_unique_ids)].tolist()
missing = set(contrib_unique_ids) - set(df.loc[contrib_idx, "unique_id"].astype(str))
if missing:
    raise ValueError(f"unique_id(s) not found in df: {missing}")

df_contrib = df.loc[contrib_idx].reset_index(drop=True)

one_hot_contrib = {
    seq_type: oh[contrib_idx]
    for seq_type, oh in one_hot_seqs.items()
}

print(f"Running contributions for {len(df_contrib)} motif(s): {contrib_unique_ids}")

Running contributions for 9 motif(s): ['2', '8', '23', '20', '25', '26', '30', '42', '49']


In [18]:
#8: Get letter plotting helper
from matplotlib.textpath import TextPath
from matplotlib.patches import PathPatch
from matplotlib import transforms
import matplotlib.font_manager as fm

def plot_weights(scores, ax, ylim=None):
    """Plot contribution scores as scaled DNA letters."""
    bases = ['A', 'C', 'G', 'T']
    colors = {'A': 'green', 'C': 'blue', 'G': 'orange', 'T': 'red'}

    for pos in range(scores.shape[0]):
        for b, base in enumerate(bases):
            val = scores[pos, b]
            if abs(val) < 1e-6:
                continue
            text = TextPath((0, 0), base, size=1,
                            prop=fm.FontProperties(family='monospace', weight='bold'))
            bbox = text.get_extents()
            t = transforms.Affine2D()
            t = t.scale(1.0 / bbox.width, val / bbox.height)
            t = t.translate(pos - 0.5, 0)
            patch = PathPatch(text.transformed(t),
                              facecolor=colors[base], linewidth=0)
            ax.add_patch(patch)

    ax.set_xlim(-1, scores.shape[0])
    if ylim:
        ax.set_ylim(ylim[0], ylim[1])
    else:
        ax.autoscale_view()
    ax.axhline(0, color='black', linewidth=0.5)

In [9]:
#8b: Format
def plot_delta_track(mut_scores, wt_scores, ax):
    """Bar chart of (mutant - wt) contribution at each position. Positive = higher in mutant."""
    delta = mut_scores.sum(axis=1) - wt_scores.sum(axis=1)
    positions = np.arange(len(delta))
    colors = np.where(delta >= 0, "red", "blue")
    ax.bar(positions, delta, color=colors, width=1.0)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel("Δ contribution\n(mutant − wt)")

In [19]:
#9: Compute contribution scores for wt & mutant sequences, per model
contributions = {}
for model_name, explainer in explainers.items():
    contributions[model_name] = {}
    for seq_type, oh in one_hot_contrib.items():
        print(f"Computing contribution scores: {model_name} / {seq_type} ...")
        shap_scores = explainer.shap_values(oh, progress_message=100)
        contributions[model_name][seq_type] = shap_scores * oh

Computing contribution scores: model_wt / wt ...
Done 0 examples of 9
Computing contribution scores: model_wt / mutant ...
Done 0 examples of 9
Computing contribution scores: model_dtag_3h / wt ...
Done 0 examples of 9
Computing contribution scores: model_dtag_3h / mutant ...
Done 0 examples of 9
Computing contribution scores: model_dtag_1d / wt ...
Done 0 examples of 9
Computing contribution scores: model_dtag_1d / mutant ...
Done 0 examples of 9


In [20]:
#10: Plot contrib
plot_dir = "/storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/"
os.makedirs(plot_dir, exist_ok=True)

model_names = ["model_wt", "model_dtag_3h", "model_dtag_1d"]
seq_types = ["wt", "mutant"]

crop = 557

for i, row in enumerate(df_contrib.itertuples()):
    uid = row.unique_id
    motif_start = row.motif_start_in_output
    motif_end   = row.motif_end_in_output

    out_start, out_end = contrib_windows[str(uid)]

    rel_motif_start = motif_start - out_start
    rel_motif_end   = motif_end - out_start

    all_scores = np.concatenate([
        contributions[m][s][i, crop + out_start:crop + out_end, :].flatten()
        for m in model_names
        for s in seq_types
    ])
    ymin, ymax = all_scores.min() * 1.1, all_scores.max() * 1.1

    fig, axes = plt.subplots(len(model_names) * 3, 1, figsize=(14, 22), sharex=True)

    panel_idx = 0
    for model_name in model_names:
        wt_scores  = contributions[model_name]["wt"][i, crop + out_start:crop + out_end, :]
        mut_scores = contributions[model_name]["mutant"][i, crop + out_start:crop + out_end, :]

        for label, scores in [("WT", wt_scores), ("MUTANT", mut_scores)]:
            ax = axes[panel_idx]
            plot_weights(scores, ax, ylim=[ymin, ymax])
            ax.axvspan(rel_motif_start, rel_motif_end, alpha=0.15, color="grey")
            ax.set_title(f"{model_name} - {label}")
            ax.set_ylabel("Contribution score")
            panel_idx += 1

        ax = axes[panel_idx]
        plot_delta_track(mut_scores, wt_scores, ax)
        ax.axvspan(rel_motif_start, rel_motif_end, alpha=0.15, color="grey")
        ax.set_title(f"{model_name} - MUTANT minus WT")
        panel_idx += 1

    axes[-1].set_xlabel("Position (bp)")

    fig.suptitle(uid)
    plt.tight_layout()
    outfile = os.path.join(plot_dir, f"{uid}_contrib.pdf")
    plt.savefig(outfile, dpi=150)
    plt.close(fig)

print(f"\nAll contribution plots written to: {plot_dir}")


All contribution plots written to: /storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/


In [21]:
#11: Plot differential for model v model
model_plot_dir = "/storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/"
os.makedirs(model_plot_dir, exist_ok=True)

model_names = ["model_wt", "model_dtag_3h", "model_dtag_1d"]
seq_types = ["wt", "mutant"]

crop = 557
baseline_model = "model_wt"          # the dmso / no-treatment model
dtag_models = ["model_dtag_3h", "model_dtag_1d"]

for i, row in enumerate(df_contrib.itertuples()):
    uid = row.unique_id
    motif_start = row.motif_start_in_output
    motif_end   = row.motif_end_in_output

    out_start, out_end = contrib_windows[str(uid)]

    rel_motif_start = motif_start - out_start
    rel_motif_end   = motif_end - out_start

    all_scores = np.concatenate([
        contributions[m][s][i, crop + out_start:crop + out_end, :].flatten()
        for m in model_names
        for s in seq_types
    ])
    ymin, ymax = all_scores.min() * 1.1, all_scores.max() * 1.1

    n_rows = len(seq_types) * (len(model_names) + len(dtag_models))
    fig, axes = plt.subplots(n_rows, 1, figsize=(14, 3.2 * n_rows), sharex=True)

    panel_idx = 0
    for seq_type in seq_types:
        baseline_scores = contributions[baseline_model][seq_type][i, crop + out_start:crop + out_end, :]

        for model_name in model_names:
            ax = axes[panel_idx]
            scores = contributions[model_name][seq_type][i, crop + out_start:crop + out_end, :]
            plot_weights(scores, ax, ylim=[ymin, ymax])
            ax.axvspan(rel_motif_start, rel_motif_end, alpha=0.15, color="grey")
            ax.set_title(f"{seq_type.upper()} - {model_name}")
            ax.set_ylabel("Contribution score")
            panel_idx += 1

        for dtag_model in dtag_models:
            dtag_scores = contributions[dtag_model][seq_type][i, crop + out_start:crop + out_end, :]
            ax = axes[panel_idx]
            plot_delta_track(dtag_scores, baseline_scores, ax)
            ax.axvspan(rel_motif_start, rel_motif_end, alpha=0.15, color="grey")
            ax.set_title(f"{seq_type.upper()} - {dtag_model} minus {baseline_model}")
            panel_idx += 1

    axes[-1].set_xlabel("Position (bp)")

    fig.suptitle(uid)
    plt.tight_layout()
    outfile = os.path.join(model_plot_dir, f"{uid}_model_compare.pdf")
    plt.savefig(outfile, dpi=150)
    plt.close(fig)

print(f"\nAll model-comparison contribution plots written to: {model_plot_dir}")


All model-comparison contribution plots written to: /storage/kaelanb/analysis/lucia/analysis/custom_seq/plots/
